In [ ]:
# ==========================================
# Responsible AI & Model Interpretation
# Customer Churn Prediction Project
# ==========================================

# =========================
# 1. Import Libraries
# =========================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import shap

# =========================
# 2. Load Dataset
# =========================

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.head())

# =========================
# 3. Data Preprocessing
# =========================

# Remove missing values
df.dropna(inplace=True)

# Convert target variable
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

# Save sensitive attributes
gender_data = df['gender']
senior_data = df['SeniorCitizen']

# Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns

le = LabelEncoder()

for col in categorical_cols:
    if col != 'customerID':
        df[col] = le.fit_transform(df[col])

# Remove customer ID
df.drop('customerID', axis=1, inplace=True)

# =========================
# 4. Train-Test Split
# =========================

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 5. Train Random Forest
# =========================

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 6. Predictions
# =========================

y_pred = model.predict(X_test)

# =========================
# 7. Model Evaluation
# =========================

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# =========================
# 8. Feature Importance
# =========================

feature_importance = pd.DataFrame({

    'Feature': X.columns,
    'Importance': model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTop Feature Importances")
print(feature_importance.head(10))

# Plot Feature Importance
plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance.head(10)
)

plt.title("Top 10 Feature Importances")
plt.show()

# =========================
# 9. SHAP Explainability
# =========================

# Create SHAP explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values
shap_values = explainer.shap_values(X_test)

# =========================
# 10. SHAP Summary Plot
# =========================

print("\nGenerating SHAP Summary Plot...")

shap.summary_plot(
    shap_values[1],
    X_test
)

# =========================
# 11. SHAP Force Plot
# =========================

print("\nGenerating SHAP Force Plot...")

shap.initjs()

shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][0],
    X_test.iloc[0]
)

# =========================
# 12. Bias Analysis
# =========================

# Add predictions back to test data
bias_df = X_test.copy()

bias_df['Actual'] = y_test.values
bias_df['Predicted'] = y_pred

# Add sensitive attributes
bias_df['gender'] = gender_data.iloc[X_test.index].values
bias_df['SeniorCitizen'] = senior_data.iloc[X_test.index].values

# =========================
# 13. Gender Bias Check
# =========================

print("\nBias Analysis by Gender")

gender_groups = bias_df.groupby('gender')

for gender, group in gender_groups:

    acc = accuracy_score(
        group['Actual'],
        group['Predicted']
    )

    print(f"\nGender: {gender}")
    print("Accuracy:", acc)

# =========================
# 14. Senior Citizen Bias Check
# =========================

print("\nBias Analysis by Senior Citizen Status")

senior_groups = bias_df.groupby('SeniorCitizen')

for senior, group in senior_groups:

    acc = accuracy_score(
        group['Actual'],
        group['Predicted']
    )

    print(f"\nSenior Citizen: {senior}")
    print("Accuracy:", acc)

# =========================
# 15. Confusion Matrix
# =========================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# =========================
# 16. Mitigation Recommendations
# =========================

print("\nMitigation Recommendations")
print("-" * 40)

print("1. Use balanced training data")
print("2. Evaluate fairness metrics regularly")
print("3. Remove biased or sensitive features")
print("4. Apply threshold adjustments")
print("5. Use explainability tools continuously")

# =========================
# 17. Final Conclusion
# =========================

print("\nProject Completed Successfully!")

print("""
Responsible AI analysis was performed using:
- Feature Importance
- SHAP Explainability
- Bias Evaluation

The Random Forest model showed good predictive
performance with relatively fair behavior across
sensitive demographic groups.
""")